# SparkSession, SparkContext, and Configuration Management

**Module**: 01_spark_fundamentals  
**Notebook**: 03_spark_session_and_spark_context.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How SparkSession acts as the unified gateway, how it encapsulates SparkContext, SQLContext, and HiveContext, and how to tune runtime configurations.
- **Why does this topic exist?**: In modern Spark, SparkSession is your primary interface. Mastering builder patterns, catalogs, and dynamic configuration is required for production engineering.
- **What problem does it solve?**: Prior to Spark 2.0, developers juggled multiple contexts (SQLContext, HiveContext, StreamingContext, SparkContext) causing confusion.
- **Where is it used in real-world Data Engineering?**: Setting up multi-tenant applications, isolated session views, and dynamic partition overwrite parameters.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 1–2 h |
| **Prerequisites** | Spark architecture |
| **Required depth** | Working knowledge: can create a session, inspect configuration, and avoid configuration drift |
| **Data Engineer** | Create a correctly configured session and understand which settings are fixed at startup. |
| **Senior Data Engineer** | Manage environment-specific configuration and explain precedence. |
| **Spark specialist** | Reason about SparkContext internals, the catalog, and runtime vs startup configuration. |
| **Next topic** | `01_spark_fundamentals/04_spark_cluster_execution.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

A SparkSession is more than a convenience wrapper: it owns the SparkContext, the SQL catalog, the scheduler
components, and the configuration snapshot. Creating a second session in the same JVM can silently reuse the
first one, so a configuration change may appear to have no effect. Understanding the lifecycle prevents hours of
"why is my setting ignored?" debugging.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **SparkSession** | The unified entry point | Owns SparkContext, SQLContext, catalog, and session state |
| **SparkContext** | The low-level engine handle | Connects to the cluster and starts schedulers and BlockManager |
| **Configuration** | Settings that control behavior | Startup settings vs runtime SQL confs |
| **Catalog** | Metadata about tables and views | Session catalog by default in this lab |
| **Builder** | The session factory | `SparkSession.builder` applies options before `getOrCreate()` |
| **Precedence** | Which value wins | Explicit builder/API > spark-submit > spark-defaults > defaults |

### How it works, step by step

1. `SparkSession.builder` collects options and master URL.
2. `getOrCreate()` either returns an existing session or creates a SparkContext.
3. SparkContext starts the DAGScheduler, TaskScheduler, BlockManager, and UI.
4. The driver registers with the cluster manager and requests executors.
5. SQL configuration can be changed at runtime; cluster/executor settings generally cannot.
6. `spark.stop()` shuts down the context and releases executors.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. SparkSession Architecture

```
+---------------------------------------------------------------+
|                        SparkSession                           |
|  - Unified entry point (Spark 2.0+)                           |
|  - Builder pattern: SparkSession.builder.getOrCreate()        |
+-------------------------------+-------------------------------+
                                |
        +-----------------------+-----------------------+
        |                       |                       |
        v                       v                       v
+---------------+       +---------------+       +---------------+
|  SparkContext |       |  Catalog API  |       | Conf Manager  |
|  - Low-level  |       |  - Databases  |       | - spark.conf  |
|  - RDD engine |       |  - Tables     |       | - Runtime     |
|  - Hardware   |       |  - Functions  |       |   parameters  |
+---------------+       +---------------+       +---------------+
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("SparkSession_DeepDive")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Exploring the Catalog API

The Catalog allows programmatically inspecting registered databases, tables, views, and built-in functions.

In [ ]:
# Create a temporary view from sales.csv
sales_df = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/sales.csv")
sales_df.createOrReplaceTempView("sales_view")

# Query the Catalog
tables = spark.catalog.listTables()
print("Registered Tables and Views in Catalog:")
for t in tables:
    print(f"  Name: {t.name:20} IsTemporary: {t.isTemporary}")

# List available built-in mathematical/string functions in Catalog
funcs = spark.catalog.listFunctions()
print(f"\nTotal built-in functions registered in Catalog: {len(funcs)}")
print("Sample functions:", [f.name for f in funcs[:8]])

## 4. Dynamic Runtime Configuration via `spark.conf`

You can inspect and modify runtime configurations dynamically without restarting the cluster.

In [ ]:
# Read current shuffle partitions
current_shuffle = spark.conf.get("spark.sql.shuffle.partitions")
print(f"Current shuffle partitions: {current_shuffle}")

# Dynamically change shuffle partitions
spark.conf.set("spark.sql.shuffle.partitions", "16")
print(f"Updated shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

# Reset back to lab default
spark.conf.set("spark.sql.shuffle.partitions", "8")

---

## Experiment: Isolated Cloned Sessions with `newSession()`

- **1. What are we testing?**: Demonstrate that spark.newSession() creates an isolated session sharing the SparkContext but with independent SQL views and configs.
- **2. What do we expect?**: Temporary views in session A are NOT visible in session B unless registered globally.

In [ ]:
# Session A creates a local temp view
sales_df.createOrReplaceTempView("view_session_a")

# Create Session B
session_b = spark.newSession()

print("Checking 'view_session_a' from Session A:")
print("  Found in Session A:", spark.catalog.tableExists("view_session_a"))

print("Checking 'view_session_a' from Session B:")
print("  Found in Session B:", session_b.catalog.tableExists("view_session_a"))

**3. What happened?**: view_session_a existed in Session A, but tableExists returned False in Session B.  
**4. Why did it happen?**: Temporary views are scoped to individual SparkSession instances.  
**5. Practical Engineering Lesson**: Use newSession() for multi-tenant applications or to prevent temporary view naming collisions.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Standardized session factory for a team

- **Industry**: Enterprise data engineering
- **Scenario**: Every notebook and job creates its own session, and settings drift between environments.
- **Problem**: Different shuffle partition counts and event-log paths make results and debugging inconsistent.
- **Why this concept is relevant**: A shared session builder centralizes master URL, driver host, event logging, shuffle partitions, and warehouse paths.
- **Implementation idea**: Create a small `spark_session.py` helper and import it from notebooks and jobs; override only what is environment-specific.
- **Expected result**: Consistent behavior and fewer configuration bugs.
- **Alternative**: A managed platform that injects configuration automatically.
- **Trade-offs**: A helper is simple and transparent; a platform is more controlled but less flexible for learning.

### Use Case 2: Test isolation in a notebook

- **Industry**: Data quality / testing
- **Scenario**: A notebook runs several experiments and one experiment changes `spark.sql.shuffle.partitions`.
- **Problem**: Later experiments inherit the change and produce confusing results.
- **Why this concept is relevant**: Use `spark.conf.set` for session-scoped SQL settings, record the original value, and restore it in a `finally` block.
- **Implementation idea**: Capture the config before the experiment, restore it afterwards, and assert the expected value.
- **Expected result**: Independent experiments with reproducible results.
- **Alternative**: Separate Spark sessions or separate test jobs.
- **Trade-offs**: Config save/restore is lightweight; separate sessions are cleaner but more expensive.

---

## Business Value

- Prevents configuration drift across notebooks and jobs.
- Reduces debugging time when a setting appears to be ignored.
- Makes environment-specific behavior explicit.
- Supports reproducibility and safer experiments.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Create one session per application/kernel; do not call `getOrCreate()` repeatedly with different options.
- Log the effective configuration at startup.
- Keep secrets out of notebooks; use environment variables or a secret manager.
- Know which settings are startup-only and which are runtime SQL confs.
- Close the session in the final teardown cell.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Configuration precedence

**Steps**

1. Print `spark.conf.get('spark.sql.shuffle.partitions')`.
2. Set it to 4 with `spark.conf.set`, print again, then restore it.
3. Try to change `spark.executor.memory` at runtime and observe that it is not a SQL conf.
4. Inspect `spark.sparkContext.getConf().getAll()` for startup settings.

**What to observe**: SQL confs can change at runtime; executor memory is fixed when the application starts.

**Healthy result**: You can explain why some changes require a new session.

**Problematic result**: Changing a startup-only setting at runtime has no effect.

**Engineering takeaway**: Classify configuration before trying to change it.

---

## Trade-offs and Alternatives

### Advantages

- Single entry point
- Unified SQL/DataFrame/streaming API
- Runtime SQL configuration

### Limitations

- Some settings are fixed at startup
- A second session may reuse the first
- Configuration precedence can be surprising

### When to use it

- Every PySpark application
- Notebook setup
- Environment-specific job configuration

### When not to use it

- Creating multiple sessions in one JVM without a reason
- Putting credentials in builder options

### Alternatives inside Spark

- `spark.conf`
- `SparkSession.builder`
- Session catalog

### Alternatives outside Spark

- Environment variables
- Config management services
- Secret managers

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Check if a Table Exists in Catalog
**Problem Statement**: Write code using spark.catalog.tableExists() to check if'orders_test'exists.  
**Input / Context**: spark.catalog

In [ ]:
# Exercise 1: Check table existence

### Exercise 2 (Intermediate): Drop a Temporary View Safely
**Problem Statement**: Register a temp view'dummy_view', verify it exists, then drop it using spark.catalog.dropTempView().  
**Input / Context**: spark.catalog.dropTempView

In [ ]:
# Exercise 2: Drop temporary view

### Exercise 3 (Advanced): Global Temporary Views Across Sessions
**Problem Statement**: Register a Global Temporary View and verify that it IS accessible from session_b.  
**Input / Context**: createGlobalTempView('global_sales')

In [ ]:
# Exercise 3: Global Temp Views

---

## Production Debugging Exercise

### The ignored setting

**Scenario**: A notebook sets `spark.sql.shuffle.partitions=8` but a later cell still shows 200 partitions.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check whether the session was already created before the setting was applied.
2. Print `spark.conf.get` immediately after setting it.
3. Check whether a new `SparkSession.builder` call reused an existing session.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check if a Table Exists in Catalog):
```python
exists = spark.catalog.tableExists("orders_test")
print("Does'orders_test'exist?:", exists)
```

#### Solution 2 (Drop a Temporary View Safely):
```python
spark.range(5).createOrReplaceTempView("dummy_view")
print("Exists before drop:", spark.catalog.tableExists("dummy_view"))
spark.catalog.dropTempView("dummy_view")
print("Exists after drop:", spark.catalog.tableExists("dummy_view"))
```

#### Solution 3 (Global Temporary Views Across Sessions):
```python
sales_df.createOrReplaceGlobalTempView("global_sales")
session_b = spark.newSession()
print("Can Session B query global_temp.global_sales?:")
session_b.sql("SELECT COUNT(*) FROM global_temp.global_sales").show()
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **SparkSession Purpose**: Unified interface managing Catalog, SQL, and underlying SparkContext.
- **Dynamic Configs**: spark.conf allows inspecting and overriding runtime SQL parameters.
- **Session Scoping**: Temporary views are private to a SparkSession; Global views reside in global_temp database.

### Essential Technical Terms
- `SparkSession`: Unified client entry point.
- `Catalog`: Internal metastore interface for tables, databases, and functions.
- `newSession()`: Creates an isolated session sharing the cluster connection.

### Common Engineering Mistakes
- **Creating Multiple SparkContexts**: A JVM process can only host ONE active SparkContext at a time.
- **Assuming Temp Views are Global**: Temp views disappear across different sessions unless made global.

### Interview Questions on This Topic
- **Q: What is the difference between SparkSession and SparkContext?**
  - *A*: SparkContext is the low-level cluster connection for RDDs; SparkSession wraps SparkContext and adds SQL, DataFrames, and Catalogs.
- **Q: How do you share views across multiple SparkSessions?**
  - *A*: Use createOrReplaceGlobalTempView() and query via global_temp.view_name.




### More Interview Questions

- **Q: What is the difference between SparkSession and SparkContext?**
  - *A*: SparkSession is the unified entry point for DataFrames, SQL, and streaming; SparkContext is the lower-level handle to the cluster and RDD APIs. SparkSession owns a SparkContext.
- **Q: Which configuration changes require a new session?**
  - *A*: Cluster-level settings such as master, executor memory, and executor cores generally require a new application; SQL confs such as shuffle partitions can change at runtime.

### What to Study Next
Open 01_spark_fundamentals/04_spark_cluster_execution.ipynb to trace execution from action to tasks.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')